# ⚖️ RAG Legal Assistant: Google Gemma + Hybrid Retrieval
**Mata Kuliah:** Natural Language Processing (UAS)

Notebook ini mengimplementasikan pipeline **Retrieval-Augmented Generation (RAG)** lengkap untuk konsultasi hukum Klinik Hukumonline.

### 🏛️ Komponen Arsitektur:
1. **Dense Retrieval:** FAISS `IndexFlatIP` dengan representasi vektor `multilingual-e5-large` (1024 dimensi)
2. **Sparse Retrieval:** BM25Okapi berbasis leksikal `title + question + summarize`
3. **Fusion:** Reciprocal Rank Fusion (RRF, $k=60$)
4. **Smart Query Rewriter:** LLM mengubah pertanyaan informal/slang user ke terminologi hukum baku sebelum retrieval
5. **Generator LLM:** Google Gemma (`google/gemma-4-26b-a4b-it:free` via OpenRouter API)

## 1. Install & Import Dependensi

In [ ]:
!pip install -q faiss-cpu rank-bm25 sentence-transformers requests python-dotenv numpy

In [ ]:
import os
import json
import requests
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

# 1. Coba baca dari Google Colab Secrets (icon kunci di sidebar Colab)
if not OPENROUTER_API_KEY:
    try:
        from google.colab import userdata
        OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        pass

# 2. Coba baca dari file .env lokal
if not OPENROUTER_API_KEY and os.path.exists(".env"):
    with open(".env", "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("OPENROUTER_API_KEY="):
                OPENROUTER_API_KEY = line.split("=", 1)[1].strip().strip('"\'')
                break

# 3. Jika belum terisi sama sekali, minta input sekali
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Masukkan OpenRouter API Key: ").strip()

print("✅ API Key siap digunakan.")

## 2. Inisialisasi Klien LLM (Google Gemma via OpenRouter)
Menggunakan model `google/gemma-4-26b-a4b-it:free` dengan fallback otomatis jika kuota publik upstream sedang sibuk.

In [ ]:
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Daftar model berurutan (prioritas utama: Google Gemma)
PRIMARY_MODELS = [
    "google/gemma-4-26b-a4b-it:free",
    "google/gemma-4-31b-it:free",
    "nvidia/nemotron-3.5-lightning:free"
]

def call_llm(messages: list[dict], temperature: float = 0.3, max_tokens: int = 1000) -> str:
    """
    Memanggil LLM via OpenRouter API dengan retry/fallback model cerdas.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/sammdavid/UAS-Natural-Language-Processing",
        "X-Title": "UAS NLP Hukumonline RAG"
    }
    
    payload = {
        "models": PRIMARY_MODELS,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    
    try:
        response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=45)
        res_data = response.json()
        
        if response.status_code != 200:
            err_msg = res_data.get("error", {}).get("message", "Unknown error")
            raise RuntimeError(f"API Error {response.status_code}: {err_msg}")
            
        content = res_data["choices"][0]["message"]["content"]
        return content.strip()
    except Exception as e:
        return f"[Error saat memanggil LLM: {str(e)}]"

# Tes konektivitas LLM
test_response = call_llm([{"role": "user", "content": "Katakan 'Koneksi ke LLM Berhasil!' dalam 1 kalimat." }])
print("Respon LLM:", test_response)

## 3. Memuat Dataset Vektor & Indeks Hybrid

In [ ]:
# 1. Load data
assert os.path.exists("vectors.npy"), "File vectors.npy tidak ditemukan!"
assert os.path.exists("documents.json"), "File documents.json tidak ditemukan!"

vectors = np.load("vectors.npy").astype(np.float32)
with open("documents.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

print(f"Data loaded: {len(docs)} dokumen, vektor dimensi: {vectors.shape[1]}")

# 2. Build FAISS (Dense)
faiss_index = faiss.IndexFlatIP(vectors.shape[1])
faiss_index.add(vectors)

# 3. Build BM25 (Sparse)
bm25_corpus = [
    (d["title"] + " " + d["question"] + " " + d["summarize"]).lower().split()
    for d in docs
]
bm25_index = BM25Okapi(bm25_corpus)

# 4. Load E5 Model
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device=device)
print(f"✅ Pipeline Retrieval siap ({device.upper()}).")

## 4. Modul Hybrid Search (FAISS + BM25 + RRF)

In [ ]:
def hybrid_search(query: str, top_k: int = 3, rrf_k: int = 60) -> list[dict]:
    """
    Melakukan pencarian dokumen relevan menggunakan perpaduan FAISS + BM25 + RRF.
    """
    # Dense encoding
    q_vec = embed_model.encode(
        f"query: {query.strip()}",
        normalize_embeddings=True,
        convert_to_numpy=True
    ).reshape(1, -1).astype(np.float32)
    
    candidate_pool = min(top_k * 10, len(docs))
    dense_scores, dense_ids = faiss_index.search(q_vec, candidate_pool)
    dense_ids = dense_ids[0].tolist()
    
    # Sparse BM25
    bm25_scores = bm25_index.get_scores(query.lower().split())
    sparse_ids = np.argsort(bm25_scores)[::-1][:candidate_pool].tolist()
    
    # RRF Scoring
    rrf_scores = {}
    for rank, doc_id in enumerate(dense_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank + 1)
    for rank, doc_id in enumerate(sparse_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank + 1)
        
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    retrieved_docs = []
    for doc_id, rrf_score in ranked:
        item = docs[doc_id].copy()
        item["rrf_score"] = round(rrf_score, 5)
        retrieved_docs.append(item)
        
    return retrieved_docs

print("Fungsi hybrid_search() siap.")

## 5. Smart Query Rewriter (Mengatasi Bahasa Gaul & Slang)
Modul ini menggunakan LLM untuk memetakan bahasa informal/slang pengguna ke **terminologi hukum baku** sebelum proses retrieval.
Ini secara otomatis memecahkan masalah ambiguitas seperti *'bos'* (atasan) vs *'Dana BOS'* (sekolah).

In [ ]:
def rewrite_query(user_query: str) -> str:
    """
    Mengubah pertanyaan informal pengguna menjadi keyword pencarian hukum baku.
    """
    system_prompt = (
        "Anda adalah asisten pencarian hukum cerdas. Tugas Anda adalah mengubah pertanyaan "
        "informal atau bahasa sehari-hari dari pengguna menjadi kata kunci hukum formal yang tepat "
        "untuk pencarian artikel hukum di Klinik Hukumonline.\n\n"
        "Aturan:\n"
        "1. Ganti kata slang/sehari-hari (contoh: 'bos' -> 'pengusaha pemberi kerja', 'cairin uang lembur' -> 'upah kerja lembur', 'duit kantor kepake' -> 'tindak pidana penggelapan jabatan').\n"
        "2. Output HANYA berupa frasa kata kunci hasil konversi singkat (3-8 kata). Tanpa tanda kutip, tanpa penjelasan tambahan."
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Pertanyaan informal: {user_query}"}
    ]
    
    rewritten = call_llm(messages, temperature=0.1, max_tokens=60)
    # Bersihkan jika ada tanda petik
    cleaned = rewritten.replace('"', '').replace("'", "").strip()
    return cleaned

# Demo perbandingan sebelum dan sesudah rewrite
query_test = "bos gamau cairin uang lembur lapor kemana ya?"
print("Query Asli     :", query_test)
print("Query Rewritten:", rewrite_query(query_test))

## 6. Pipeline RAG Lengkap (End-to-End Chatbot Konsultasi Hukum)
Menggabungkan: **User Query** ➔ **Query Rewriter** ➔ **Hybrid Retrieval** ➔ **Prompt Augmentation** ➔ **Gemma Synthesis**.

In [ ]:
def rag_chat(user_question: str, use_query_rewriter: bool = True, top_k: int = 3) -> dict:
    """
    Menjalankan alur RAG lengkap untuk menjawab konsultasi hukum pengguna.
    """
    # 1. Query Reformulation (opsional tapi sangat direkomendasikan)
    search_query = user_question
    rewritten_text = None
    if use_query_rewriter:
        rewritten_text = rewrite_query(user_question)
        search_query = rewritten_text
        
    # 2. Retrieval Dokumen Konteks
    retrieved_docs = hybrid_search(search_query, top_k=top_k)
    
    # 3. Format Konteks untuk Prompt LLM
    context_parts = []
    for i, d in enumerate(retrieved_docs, 1):
        # Batasi ulasan jawaban dokumen ~1500 karakter agar konteks tetap padat & relevan
        answer_snippet = d.get('answer', '')[:1500].strip()
        context_parts.append(
            f"[DOKUMEN RUJUKAN {i}]\n"
            f"Judul: {d['title']}\n"
            f"Ringkasan Masalah: {d['summarize']}\n"
            f"Penjelasan Hukum: {answer_snippet}\n"
            f"Sumber: {d['source']}\n"
        )
    context_str = "\n---\n".join(context_parts)
    
    # 4. Susun System Prompt Konsultan Hukum Profesional
    system_instruction = (
        "Anda adalah Konsultan Hukum AI profesional dari Klinik Hukumonline.\n"
        "Tugas Anda adalah memberikan jawaban hukum yang akurat, jelas, empatik, dan terstruktur berdasarkan dokumen rujukan yang dilampirkan.\n\n"
        "Panduan Menjawab:\n"
        "1. Berikan kesimpulan/jawaban lugas di awal.\n"
        "2. Sertakan Dasar Hukum (Pasal/UU/Peraturan) yang disebutkan di dokumen rujukan.\n"
        "3. Jelaskan langkah praktis atau jalur hukum yang dapat ditempuh pengguna.\n"
        "4. Cantumkan tautan rujukan artikel Klinik Hukumonline di akhir jawaban.\n"
        "5. Jika dokumen rujukan tidak mencukupi, sampaikan secara jujur tanpa halusinasi."
    )
    
    user_prompt = (
        f"Pertanyaan Pengguna:\n{user_question}\n\n"
        f"Berikut adalah Dokumen Rujukan Resmi dari Klinik Hukumonline:\n"
        f"{context_str}\n\n"
        f"Berdasarkan dokumen rujukan di atas, mohon berikan nasihat hukum yang komprehensif:"
    )
    
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_prompt}
    ]
    
    # 5. Sintesis Jawaban oleh Gemma
    llm_answer = call_llm(messages, temperature=0.2, max_tokens=1200)
    
    return {
        "question": user_question,
        "rewritten_query": rewritten_text,
        "retrieved_docs": retrieved_docs,
        "answer": llm_answer
    }

print("Fungsi rag_chat() siap digunakan.")

## 7. Uji Coba Komparasi: Mengapa Query Rewriter Mengubah Segalanya? 🚀
Mari kita buktikan pada pertanyaan sulit: `"bos gamau cairin uang lembur lapor kemana ya?"`.

In [ ]:
test_query = "bos gamau cairin uang lembur lapor kemana ya?"

print("=== [HASIL 1] RAG DENGAN SMART QUERY REWRITER ===\n")
res_smart = rag_chat(test_query, use_query_rewriter=True, top_k=3)
print("-> Query ditransformasikan menjadi:", res_smart["rewritten_query"])
print("-> Rujukan Dokumen Teratas:")
for i, doc in enumerate(res_smart["retrieved_docs"], 1):
    print(f"   [{i}] {doc['title']} (URL: {doc['source']})")
    
print("\n=== JAWABAN GEMMA ===\n")
print(res_smart["answer"])

## 8. Uji Kasus-Kasus Hukum Lainnya

In [ ]:
# Kasus 2: Sengketa Pertanahan Girik
res_tanah = rag_chat("bagaimana cara membeli tanah girik tanpa sertifikat yang aman?")
print("=== KASUS TANAH GIRIK ===\n")
print(res_tanah["answer"])

In [ ]:
# Kasus 3: Cross-lingual English Query
res_cross = rag_chat("how can a foreigner legally own property or land in bali?")
print("=== KASUS KEPEMILIKAN PROPERTI WNA ===\n")
print(res_cross["answer"])

## 9. Konsultasi Interaktif Bebas
Jalankan cell ini untuk mengajukan pertanyaan hukum apa saja secara langsung.

In [ ]:
user_input = input("Ketikkan pertanyaan hukum Anda: ")
if user_input.strip():
    result = rag_chat(user_input.strip())
    print("\n" + "="*60)
    print("Pertanyaan :", result["question"])
    if result["rewritten_query"]:
        print("Terminologi:", result["rewritten_query"])
    print("="*60 + "\n")
    print(result["answer"])
    print("\nDokumen Sumber:")
    for d in result["retrieved_docs"]:
        print(f"- {d['title']}: {d['source']}")